# RWB Timestral

In [ ]:
%pip install -q numpy==2.5.4 pandas==2.2.3 pyarrow==26.0.1

In [ ]:
from base64 import b64decode
from importlib.util import find_spec
from io import BytesIO
from pathlib import Path
from subprocess import run
from tempfile import TemporaryDirectory
from time import perf_counter
from zipfile import ZipFile
import json
import os
import sys

import numpy as np
import pandas as pd

PROJECT_DIR = None
RUN = "saved"
DEVICE = "auto"
THREADS = 7

H = 8

MODEL = "mistralai/Timestral-209M"
REVISION = "9874d252ab0aae16f7f7c999f98f98d6ad7d8365"
SOURCE_REVISION = "a534bdbd244d64f3d9d743f46c7bf6fc330ed4ff"
CONTEXT, DAY = 336, 48
QUANTILES = (.52, .53, .54, .55, .56, .57, .58)
BASE = {"recipe": "history", "quantiles": [.55, .55]}
RECIPES = ("history", "weekly", "analogue_1", "analogue_3", "hourly_phases")
TUNING = ("2025-08-30", "2025-09-06", "2025-09-13", "2025-09-16")
CONTROL = ("2025-09-20", "2025-09-27", "2025-10-17", "2025-10-19",
           "2025-10-20", "2025-10-22", "2025-10-24", "2025-10-26")

CHECKPOINT = os.environ.get("TIMESTRAL_CHECKPOINT", MODEL)

DRIVE_MOUNT = Path("/content/drive")


def is_project(path):
    return (path / "data").is_dir() and (path / "solutions").is_dir()


def resolve_project():
    cwd = Path.cwd().resolve()
    requested = None if PROJECT_DIR is None else Path(PROJECT_DIR).expanduser()
    if requested is None:
        for folder in (cwd, *cwd.parents):
            if is_project(folder):
                return folder, folder / "solutions"
    elif is_project(requested):
        folder = requested.resolve()
        return folder, folder / "solutions"

    try:
        from google.colab import drive
    except ImportError:
        if requested is not None:
            raise FileNotFoundError(f"В {requested} должны быть папки data и solutions.")
        folder = cwd.parent if cwd.name == "solutions" else cwd
        return folder, folder / "solutions"

    drive_root = DRIVE_MOUNT / "MyDrive"
    if not drive_root.is_dir():
        drive.mount(str(DRIVE_MOUNT))

    if requested is not None:
        folder = requested if requested.is_absolute() else drive_root / requested
        if not is_project(folder):
            raise FileNotFoundError(f"В {folder} должны быть папки data и solutions.")
        return folder, folder / "solutions"

    candidates = [folder for folder in (drive_root, *drive_root.iterdir()) if is_project(folder)]
    named = [folder for folder in candidates
             if (folder / "solutions/full_timestral_LLM_solution.ipynb").is_file()]
    matches = named or candidates
    if len(matches) == 1:
        return matches[0], matches[0] / "solutions"
    raise FileNotFoundError("Задайте PROJECT_DIR путём к папке с data и solutions на Google Диске.")


ROOT, SOLUTIONS = resolve_project()
DATA_DIR = Path(os.environ.get("DATA_DIR", ROOT / "data")).expanduser().resolve()
OUT = SOLUTIONS / "timestral"
SUBMISSION = SOLUTIONS / "full_timestral_LLM_solution.csv"

## Данные

Папки `data/` и `solutions/` лежат рядом. В Colab при необходимости задайте `PROJECT_DIR`.

In [ ]:
_assets_directory = TemporaryDirectory(prefix="rwb-timestral-")
ASSETS = Path(_assets_directory.name)
with ZipFile(BytesIO(b64decode(
    b"..."
))) as archive:
    archive.extractall(ASSETS)

SAVED = ASSETS / "results"
TEST_PATH = DATA_DIR / "test_solo_track.parquet"
if not TEST_PATH.is_file():
    TEST_PATH = ASSETS / "data/test_solo_track.parquet"

In [ ]:
class History:
    def __init__(self, folder, use_status, test_path=None):
        columns = ["route_id", "timestamp", "target_1h"]
        if use_status:
            columns.append("status_3")
        train = pd.read_parquet(folder / "train_solo_track.parquet", columns=columns)
        train = train.sort_values(["route_id", "timestamp"])
        self.routes = train.route_id.unique()
        self.times = pd.DatetimeIndex(train.timestamp.unique()).sort_values()
        shape = (len(self.routes), len(self.times))
        if (len(train) != np.prod(shape)
                or train.duplicated(["route_id", "timestamp"]).any()
                or not (np.diff(self.times.asi8) == 30 * 60 * 10**9).all()):
            raise ValueError("Ожидается полная получасовая сетка маршрутов.")
        self.y = train.target_1h.to_numpy(dtype=np.float32).reshape(shape)
        if not np.isfinite(self.y).all() or (self.y < 0).any():
            raise ValueError("В истории отгрузок ожидаются конечные неотрицательные значения.")
        self.status = train.status_3.to_numpy(dtype=np.float32).reshape(shape) if use_status else None
        self.test = pd.read_parquet(TEST_PATH if test_path is None else test_path)
        self.future = pd.DatetimeIndex(self.test.timestamp.unique()).sort_values()
        expected = pd.date_range(self.times[-1] + pd.Timedelta(minutes=30), periods=H, freq="30min")
        if (not self.future.equals(expected) or self.future[0].strftime("%H:%M") != "11:00"
                or not self.test.id.is_unique or self.test.duplicated(["route_id", "timestamp"]).any()
                or len(self.test) != len(self.routes) * H
                or set(self.test.route_id) != set(self.routes)):
            raise ValueError("Тест должен продолжать историю на восемь шагов с 11:00.")

    def origin(self, date):
        if date == "test":
            return len(self.times)
        return self.times.get_loc(pd.Timestamp(date) + pd.Timedelta(hours=11))


def metric(actual, prediction):
    y = actual.astype(np.float64)
    p = prediction.astype(np.float64)
    volume = y.sum()
    bias = (p.sum() - volume) / volume
    wape = np.abs(p - y).sum() / volume
    return {"score": wape + abs(bias), "wape": wape, "signed_bias": bias,
            "wape_5_8": np.abs(p[:, 4:] - y[:, 4:]).sum() / y[:, 4:].sum()}


def write_submission(data, forecast, output):
    lookup = pd.DataFrame({"route_id": np.repeat(data.routes, H),
                           "timestamp": np.tile(data.future.to_numpy(), len(data.routes)),
                           "y_pred": forecast.ravel()})
    submission = data.test[["id", "route_id", "timestamp"]].merge(
        lookup, on=["route_id", "timestamp"], validate="one_to_one",
    )[["id", "y_pred"]]
    assert len(submission) == 8000 and submission.id.equals(data.test.id)
    assert np.isfinite(submission.y_pred).all() and submission.y_pred.ge(0).all()
    output.parent.mkdir(parents=True, exist_ok=True)
    temp = output.with_suffix(".tmp")
    submission.to_csv(temp, index=False)
    temp.replace(output)
    print(f"{output} · {len(submission):,} строк · сумма {forecast.sum(dtype=np.float64):,.0f}", flush=True)

## Модель

In [ ]:
def recent_distance(panel, origin, candidates):
    offsets = np.arange(-4, 0)
    present = panel[:, origin+offsets]
    previous = panel[:, candidates[:, None]+offsets]
    scale = np.maximum(panel[:, origin-CONTEXT:origin].mean(axis=1), 1.)
    level = np.abs(previous-present[:, None, :]) @ np.array([.1, .2, .3, .4])
    slope = np.abs(np.diff(previous, axis=-1) - np.diff(present, axis=-1)[:, None, :]).mean(axis=-1)
    return (.75*level + .25*slope) / scale[:, None]


def historical_analogues(data, origin, count):


    candidates = np.arange(origin-DAY, CONTEXT-1, -DAY, dtype=int)
    assert (candidates + H <= origin).all()
    current_day = (data.times[-1].dayofweek if origin == len(data.times)
                   else data.times[origin].dayofweek)
    weekdays = data.times[candidates].dayofweek.to_numpy()
    same_day = weekdays == current_day
    if same_day.sum() >= 3:
        candidates = candidates[same_day]
    else:
        candidates = candidates[(weekdays >= 5) == (current_day >= 5)]
    if len(candidates) < count:
        raise ValueError("Недостаточно прошлых сопоставимых дней для подбора аналога.")

    target_distance = recent_distance(data.y, origin, candidates)
    status_distance = recent_distance(data.status, origin, candidates)
    recency = .015 * (origin-candidates) / (7*DAY)
    distance = .6*target_distance + .4*status_distance + recency
    matched = candidates[np.argsort(distance, axis=1, kind="stable")[:, :count]]
    return matched


def inputs(data, origin, recipe):
    context = data.y[:, origin-CONTEXT:origin]
    if recipe == "history":
        return context, None, H
    if recipe == "hourly_phases":
        return np.stack((context[:, ::2], context[:, 1::2]), axis=1), None, H//2
    if recipe == "weekly":
        lag = origin - 7*DAY
        analogue = data.y[:, lag-CONTEXT:lag+H]
    else:
        count = int(recipe.rsplit("_", 1)[1])
        origins = historical_analogues(data, origin, count)
        routes = np.arange(len(context))[:, None, None]
        history = data.y[routes, origins[:, :, None]+np.arange(-CONTEXT, H)]
        analogue = np.median(history, axis=1).astype(np.float32)
    return context, analogue[:, None, :], H


def read_quantile(forecast, q):
    levels = forecast.quantile_levels
    i = int(np.searchsorted(levels, q))
    if levels[i] == q:
        return forecast.quantile(q)
    lo, hi = levels[i-1:i+1]
    return torch.lerp(forecast.quantile(lo), forecast.quantile(hi), (q-lo)/(hi-lo))


def predict(model, data, origin, recipe, use_bf16):
    x, known, horizon = inputs(data, origin, recipe)
    batch_size = 64 if known is None else 32
    forecasts = []
    with torch.inference_mode(), torch.autocast(model.device.type, dtype=torch.bfloat16, enabled=use_bf16):
        for left in range(0, len(x), batch_size):
            block = slice(left, left+batch_size)
            forecast = model.forecast(x[block], prediction_length=horizon,
                                      known_covariates=None if known is None else known[block])
            p = torch.stack([read_quantile(forecast, q) for q in QUANTILES], dim=1)

            p = p.transpose(-2, -1).reshape(len(x[block]), len(QUANTILES), H)
            forecasts.append(p.float().cpu().numpy())
    p = np.concatenate(forecasts)
    if p.shape != (len(data.routes), len(QUANTILES), H) or not np.isfinite(p).all():
        raise ValueError("Timestral вернул некорректные прогнозы.")
    return np.maximum(p, 0)


def cache_prediction(model, data, date, recipe, use_bf16, cache):
    suffix = "bf16" if use_bf16 else "fp32"
    path = cache / f"{recipe}_{date}_{suffix}.npy"
    origin = data.origin(date)
    started = perf_counter()
    if path.exists():
        p = np.load(path, allow_pickle=False)
    else:
        p = predict(model, data, origin, recipe, use_bf16)
        temporary = path.with_suffix(".tmp.npy")
        np.save(temporary, p, allow_pickle=False)
        temporary.replace(path)
    y = data.y[:, origin:origin+H]
    print(f"{date} · {recipe:>13} · q=.55 {metric(y,p[:,QUANTILES.index(.55)])['score']:.6f} · "
          f"{perf_counter()-started:.1f} с", flush=True)
    return y, p


def point(predictions, levels):
    left, right = (QUANTILES.index(q) for q in levels)
    return np.concatenate((predictions[:, left, :4], predictions[:, right, 4:]), axis=1)


def average(rows):
    return {key: float(np.mean([r[key] for r in rows.values()])) for key in next(iter(rows.values()))}


def selection_score(rows, dates):
    saturday = {date: rows[date] for date in dates if date in TUNING[:3]}
    return .5*average(rows)["score"] + .5*average(saturday)["score"]


def fit_choice(recipe, caches, dates):


    grid = [(q1,q2) for q1 in QUANTILES for q2 in QUANTILES if abs(q1-q2) <= .031]
    trials = []
    for q1,q2 in grid:
        rows = {d: metric(y, point(p, (q1,q2))) for d,(y,p) in caches.items()}
        score = selection_score(rows, dates)
        penalty = .03 * (abs(q1-.55)+abs(q2-.55))
        trials.append({"recipe": recipe, "quantiles": [q1,q2], "objective": score+penalty,
                       "score": score, "days": rows})
    return min(trials, key=lambda row: row["objective"])


def tune(model, data, cache, use_bf16):
    early = {}
    for recipe in RECIPES:
        early[recipe] = {d: cache_prediction(model, data, d, recipe, use_bf16, cache) for d in TUNING[:2]}
    screening = {recipe: fit_choice(recipe, rows, TUNING[:2]) for recipe,rows in early.items()}
    shortlist = sorted(RECIPES[1:], key=lambda recipe: screening[recipe]["objective"])[:2]
    fixed_q = BASE["quantiles"]


    def fixed_score(recipe, date):
        y, p = early[recipe][date]
        return metric(y, point(p, fixed_q))["score"]

    fixed_wins = [recipe for recipe in RECIPES[1:]
                  if all(fixed_score(recipe,d) < fixed_score("history",d) for d in TUNING[:2])]
    if fixed_wins:
        stable = min(fixed_wins, key=lambda recipe: np.mean([fixed_score(recipe,d) for d in TUNING[:2]]))
        if stable not in shortlist:
            shortlist.append(stable)
    print("Варианты после двух суббот", shortlist, flush=True)
    trials = []
    fixed_trials = []
    for recipe in ("history", *shortlist):
        rows = early[recipe]
        rows.update({d: cache_prediction(model,data,d,recipe,use_bf16,cache) for d in TUNING[2:]})
        trials.append(fit_choice(recipe, rows, TUNING))
        days = {d: metric(y, point(p, fixed_q)) for d,(y,p) in rows.items()}
        fixed_trials.append({"recipe": recipe, "quantiles": fixed_q, "days": days,
                             "score": selection_score(days, TUNING)})
    trials.sort(key=lambda row: row["objective"])
    challenger = next(t for t in trials if (t["recipe"], t["quantiles"]) != (BASE["recipe"], fixed_q))
    fixed_challenger = min(fixed_trials, key=lambda row: row["score"])
    print("Контрольный кандидат", json.dumps(challenger), flush=True)
    print("Кандидат без смены квантилей", json.dumps(fixed_challenger), flush=True)
    return {"dates": TUNING, "screening": screening, "shortlist": shortlist,
            "trials": trials, "fixed_trials": fixed_trials,
            "challenger": challenger, "fixed_challenger": fixed_challenger}


def control(model, data, cache, use_bf16, tuning):
    challenger = tuning["challenger"]
    configs = {"baseline": BASE,
               "challenger": {key: challenger[key] for key in ("recipe", "quantiles")}}
    fixed = {key: tuning["fixed_challenger"][key] for key in ("recipe", "quantiles")}
    if fixed not in configs.values():
        configs["fixed"] = fixed
    days = {key: {} for key in configs}
    for date in CONTROL:
        per_date = {}
        for name, config in configs.items():
            recipe = config["recipe"]
            if recipe not in per_date:
                per_date[recipe] = cache_prediction(model, data, date, recipe, use_bf16, cache)
            y, p = per_date[recipe]
            days[name][date] = metric(y, point(p, config["quantiles"]))
            print(f"{date} · {name} · итог {days[name][date]['score']:.6f}", flush=True)
    mean = {name: average(rows) for name, rows in days.items()}
    sat = {name: average({d: rows[d] for d in CONTROL[:2]}) for name, rows in days.items()}
    selected = min(mean, key=lambda name: mean[name]["score"])
    return {"model": MODEL, "revision": REVISION, "source_revision": SOURCE_REVISION,
            "precision": "bf16" if use_bf16 else "fp32", "context": CONTEXT,
            "configs": configs, "days": days, "mean": mean, "saturday": sat,
            "selected": selected, "leaderboard_score": None}

In [ ]:
def load_timestral():
    if find_spec("timestral") is None:
        package = "timestral @ git+https://github.com/mistralai/timestral.git@" + SOURCE_REVISION
        run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
    global torch
    import torch
    from timestral import TimestralForecaster

    torch.set_num_threads(THREADS)
    if torch.get_num_interop_threads() != 1:
        try:
            torch.set_num_interop_threads(1)
        except RuntimeError:
            pass
    device = DEVICE
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    use_bf16 = device == "cpu" or torch.cuda.is_bf16_supported()
    kwargs = {"revision": REVISION} if CHECKPOINT == MODEL else {}
    return TimestralForecaster.from_pretrained(CHECKPOINT, device=device, **kwargs), use_bf16


def export_cached():
    test = pd.read_parquet(TEST_PATH, columns=["id"])
    payload = (SAVED / "submission_100_model.csv").read_bytes()
    prediction = pd.read_csv(BytesIO(payload))
    assert prediction.columns.tolist() == ["id", "y_pred"] and len(prediction) == 8000
    assert prediction.id.equals(test.id) and prediction.id.is_unique
    assert np.isfinite(prediction.y_pred).all() and prediction.y_pred.ge(0).all()
    SUBMISSION.parent.mkdir(parents=True, exist_ok=True)
    temporary = SUBMISSION.with_suffix(".tmp")
    temporary.write_bytes(payload)
    temporary.replace(SUBMISSION)


def read_result(name):
    path = OUT / name
    if RUN == "saved" or not path.is_file():
        path = SAVED / name
    return json.loads(path.read_text(encoding="utf-8"))

## Запуск

`saved` — готовый CSV, `predict` — новый прогноз, `control` — проверка, `tune` — подбор и проверка.

In [ ]:
if RUN not in {"saved", "predict", "control", "tune"}:
    raise ValueError("RUN должен быть saved, predict, control или tune.")

if RUN == "saved":
    report = read_result("report.json")
    export_cached()
else:
    model, use_bf16 = load_timestral()
    if RUN in {"control", "tune"}:
        cache = OUT / "cache"
        cache.mkdir(parents=True, exist_ok=True)
        if RUN == "tune":
            tuning = tune(model, History(DATA_DIR, use_status=True), cache, use_bf16)
            (OUT / "tuning.json").write_text(json.dumps(tuning, indent=2), encoding="utf-8")
        else:
            tuning = read_result("tuning.json")
        use_status = any(tuning[key]["recipe"].startswith("analogue")
                         for key in ("challenger", "fixed_challenger"))
        report = control(model, History(DATA_DIR, use_status=use_status), cache, use_bf16, tuning)
        (OUT / "report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    else:
        report = read_result("report.json")
    config = report["configs"][report["selected"]]
    history = History(DATA_DIR, use_status=config["recipe"].startswith("analogue"))
    forecast = predict(model, history, history.origin("test"), config["recipe"], use_bf16)
    write_submission(history, point(forecast, config["quantiles"]), SUBMISSION)

print(SUBMISSION)
scores = pd.DataFrame({
    name: {date: row["score"] for date, row in rows.items()}
    for name, rows in report["days"].items()
})
display(scores.round(6))